# Compiler experiment log: targets & SABRE tuning vs. depth/gate count

**Pack:** `samples/research/` (#190) — literate notebook, paired with
[`compiler_experiment_log_smoke.py`](./compiler_experiment_log_smoke.py).

**Question:** how much do (a) the target's qubit topology and (b) SABRE's
lookahead window move a compiled circuit's headline metrics (`depth`,
`gate_count`)?

**Linked canonical source:** [`test/verify/qaoa.qn`](../../test/verify/qaoa.qn)
— a K4 all-to-all QAOA cost layer (see its own header comment). This fixture
already has a `ci: smoke` catalog row
(`visualization/dense-swap-mismatch`, #189) that typechecks it; this notebook
does not fork a copy, it narrates the *same* file with a wider experiment
grid than #189's fixed before/after pair.

**Not a claim about this circuit's correctness** — that's
`test/verify/qaoa.py`'s job in `just ci-rust`. This is a compiler-behavior
study: same verified program, different compiler configuration knobs.

## Setup

```bash
cargo build --release -p quonc
export QUONC=$PWD/target/release/quonc
```

Every cell below is a real `quonc` invocation you can run yourself; this
notebook ships with no fabricated outputs (`execution_count: null`,
`outputs: []` throughout — see
[`repro_appendix_template.md`](./repro_appendix_template.md) for why). The
numbers quoted in the prose come from running
[`compiler_experiment_log_smoke.py`](./compiler_experiment_log_smoke.py)
against commit `d26723141d5799a0e8107915b16c55ef3b9fa6f3`.

## Experiment 1: topology constraint (no target vs. a real 5-qubit line)

`qaoa.qn`'s cost layer touches all $\binom{4}{2}=6$ pairs among 4 logical
qubits. The generic (untargeted) backend has no adjacency constraint at all;
`targets/ibm/fake_manila_v2.json` is a 5-qubit line, so most of those pairs
are not adjacent and SABRE must insert routing (visible as extra CNOTs, not
as a nonzero `swap_count` — see the caveat below).

In [ ]:
$QUONC --metrics-json - -q test/verify/qaoa.qn
# depth=11 gate_count=15 (no topology constraint at all)

In [ ]:
$QUONC --target targets/ibm/fake_manila_v2.json \
       --metrics-json - -q test/verify/qaoa.qn
# depth=23 gate_count=45 (default --sabre-lookahead 20)

**Result:** depth more than doubles (11 -> 23) and gate count triples
(15 -> 45) once the line topology forces routing. Topology constraints are a
real, measurable compiler cost on this fixture — not a rounding artifact.

## Experiment 2: SABRE lookahead window

SABRE's `--sabre-lookahead` (SPEC §7.4) controls how far ahead the router
looks when picking which SWAP-equivalent CNOT sequence to insert next. A
narrower window is cheaper to compute but can commit to locally-good, 
globally-worse choices.

In [ ]:
$QUONC --target targets/ibm/fake_manila_v2.json --sabre-lookahead 1 \
       --metrics-json - -q test/verify/qaoa.qn
# depth=26 gate_count=48

In [ ]:
$QUONC --target targets/ibm/fake_manila_v2.json --sabre-lookahead 20 \
       --metrics-json - -q test/verify/qaoa.qn
# depth=23 gate_count=45 (quonc's default lookahead; same as Experiment 1's second cell)

**Result:** widening the lookahead window from 1 to the default (20)
recovers 3 fewer depth layers and 3 fewer gates on this fixture. The effect
is real but modest at this circuit size — see
[`na_resource_study.ipynb`](./na_resource_study.ipynb) for a case (NA
routing-awareness) where a similarly-motivated heuristic has *no* benefit,
or even a small cost.

### Log table

| Config | `--target` | `--sabre-lookahead` | `depth` | `gate_count` |
| --- | --- | --- | --- | --- |
| A: unconstrained | (none) | n/a | 11 | 15 |
| B: manila, narrow lookahead | `fake_manila_v2.json` | 1 | 26 | 48 |
| C: manila, default lookahead | `fake_manila_v2.json` | 20 | 23 | 45 |

### Caveat: `swap_count` stays 0 throughout

`--metrics-json`'s `swap_count` field only counts a literal `SWAP` op; by the
time metrics run, SABRE's routing has already been decomposed to CNOTs (see
[`samples/visualization/refresh_goldens.sh`](../visualization/refresh_goldens.sh)'s
identical caveat for this same fixture). Every cell above reports
`swap_count: 0` even though B and C clearly routed — read `gate_count`
growth, not `swap_count`, as the routing signal.

## Repro appendix

- **Quon commit:** `d26723141d5799a0e8107915b16c55ef3b9fa6f3` (branch
  `samples/190-research-notebooks`)
- **`quonc` version:** `quonc 0.1.0`
- **Build:** `cargo build --release -p quonc`
- **Python:** none required — this notebook's smoke twin is pure `stdlib`
  (`json`, `subprocess`) plus `quonc` itself
- **Smoke twin:** `python samples/research/compiler_experiment_log_smoke.py`
- **Linked canonical sources:** `test/verify/qaoa.qn`
  (`visualization/dense-swap-mismatch`, #189)